In [ ]:
!pip install transformers imbalanced-learn

import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm
import joblib

from transformers import BertTokenizer, BertModel

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report, average_precision_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier

In [ ]:
df = pd.read_csv("/content/FINAL DATASET.csv")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['clean_text'].apply(clean_text)
df = df[['clean_text', 'label']].dropna()

texts = df['clean_text'].tolist()
y = df['label'].values

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def mean_pooling(outputs, attention_mask):
    token_embeddings = outputs.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1)

def get_bert_embeddings(texts, batch_size=16):
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = bert(**inputs)

            pooled = mean_pooling(outputs, inputs['attention_mask'])
            embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings)

X_bert = get_bert_embeddings(texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 987/987 [04:32<00:00,  3.62it/s]


In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(texts).toarray()

In [ ]:
X = np.hstack((X_bert, X_tfidf))
print("Final Feature Shape:", X.shape)

Final Feature Shape: (15780, 5768)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=10,
    random_state=42
)

mlp.fit(X_train_sm, y_train_sm)

MLPClassifier(early_stopping=True, hidden_layer_sizes=(256, 128, 64),
              max_iter=500, random_state=42)

In [ ]:
probs = mlp.predict_proba(X_test_s)[:, 1]

In [ ]:
best_thresh = 0.5
best_diff = float("inf")

target_recall = 0.88

for t in np.linspace(0.1, 0.5, 50):
    preds = (probs >= t).astype(int)
    r = recall_score(y_test, preds)

    diff = abs(r - target_recall)

    if diff < best_diff:
        best_diff = diff
        best_thresh = t

print("Final Threshold:", best_thresh)

y_pred = (probs >= best_thresh).astype(int)

--- High Recall Optimization ---
Target Recall: 88.0%
Optimized Threshold: 0.0002
New Recall: 88.31%
New Precision: 36.96% (Expect this to drop, which is okay!)


['best_threshold.joblib']

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probs)
pr_auc = average_precision_score(y_test, probs)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)
balanced_acc = (recall + specificity) / 2

print(f"Accuracy           : {accuracy*100:.2f}%")
print(f"Balanced Accuracy  : {balanced_acc*100:.2f}%")
print(f"Recall             : {recall*100:.2f}%")
print(f"Precision          : {precision*100:.2f}%")
print(f"Specificity        : {specificity*100:.2f}%")
print(f"F1 Score           : {f1*100:.2f}%")
print(f"ROC-AUC            : {roc_auc:.4f}")
print(f"PR-AUC             : {pr_auc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy           : 98.35%
Balanced Accuracy  : 88.66%
Recall             : 77.92%
Precision          : 86.96%
Specificity        : 99.40%
F1 Score           : 82.19%
ROC-AUC            : 0.9597
PR-AUC             : 0.8621

Classification Report:

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      3002
         1.0       0.87      0.78      0.82       154

    accuracy                           0.98      3156
   macro avg       0.93      0.89      0.91      3156
weighted avg       0.98      0.98      0.98      3156



In [ ]:
joblib.dump(mlp, "mlp_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(tfidf, "tfidf.joblib")
joblib.dump(best_thresh, "threshold.joblib")

['threshold.joblib']

In [ ]:
def predict_text(text):
    text_clean = clean_text(text)

    # BERT
    emb_bert = get_bert_embeddings([text_clean])

    # TF-IDF
    emb_tfidf = tfidf.transform([text_clean]).toarray()

    # Combine
    emb = np.hstack((emb_bert, emb_tfidf))
    emb_s = scaler.transform(emb)

    prob = mlp.predict_proba(emb_s)[0][1]
    label = "Fraud" if prob >= best_thresh else "Real"

    return label, float(prob)

In [ ]:
from google.colab import files

files.download("mlp_model.joblib")
files.download("scaler.joblib")
files.download("tfidf.joblib")
files.download("threshold.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>